# Data Workflow — STATS19 Road Safety Collision Data

**Author:** Simon

## Setup

**Dataset:** STATS19 Road Safety Collision Data — the UK Department for Transport's
official road traffic collision database, using their pre-curated "last 5 years"
extract (`dft-road-casualty-statistics-collision-last-5-years.csv`), covering
collision years 2021-2025.

**Project description:** This notebook ingests, cleans, and explores the STATS19
collision-level dataset to understand how collision severity (Fatal / Serious /
Slight) relates to *when* a collision happens (year, hour of day, day of week,
month) and the *speed environment* it happens in (posted speed limit). It follows
a professional, reproducible data workflow: memory-conscious ingestion, explicit
cleaning functions with documented justification, a parameterised EDA function,
and a small set of interpreted visualizations — establishing a reusable
foundation ahead of later modelling work (out of scope here; see the README's
Reflections section).

## Data Ingestion

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


Only the columns needed for this analysis are loaded (`usecols`), and
coded categorical fields are read straight into pandas' `category` dtype
instead of the default `int64`/`object` inference. With ~500k rows and 37+
raw columns, loading everything at default dtypes would use several times
more memory than necessary for columns that only ever hold a handful of
distinct values.

In [2]:
RAW_PATH = "data/raw/dft-road-casualty-statistics-collision-last-5-years.csv"

dtype_map = {
    "collision_index": "string",
    "collision_year": "int16",
    "collision_severity": "category",
    "number_of_vehicles": "int8",
    "number_of_casualties": "int8",
    "day_of_week": "category",
    "speed_limit": "int8",
}

usecols = [
    "collision_index",
    "collision_year",
    "date",
    "time",
    "day_of_week",
    "collision_severity",
    "speed_limit",
    "number_of_vehicles",
    "number_of_casualties",
]

df = pd.read_csv(RAW_PATH, usecols=usecols, dtype=dtype_map)
df.info(memory_usage="deep")


<class 'pandas.DataFrame'>
RangeIndex: 513801 entries, 0 to 513800
Data columns (total 9 columns):
 #   Column                Non-Null Count   Dtype   
---  ------                --------------   -----   
 0   collision_index       513801 non-null  string  
 1   collision_year        513801 non-null  int16   
 2   collision_severity    513801 non-null  category
 3   number_of_vehicles    513801 non-null  int8    
 4   number_of_casualties  513801 non-null  int8    
 5   date                  513801 non-null  str     
 6   day_of_week           513801 non-null  category
 7   time                  513801 non-null  str     
 8   speed_limit           513801 non-null  int8    
dtypes: category(2), int16(1), int8(3), str(2), string(1)
memory usage: 89.2 MB


The data guide is loaded and filtered down to `table == "collision"` —
the workbook covers all three STATS19 tables (collision/vehicle/casualty), and
only the collision-table rows are relevant lookups for this dataset.

In [3]:
DATA_GUIDE_PATH = "data/dimensions/dft-road-casualty-statistics-road-safety-open-dataset-data-guide-2025.xlsx"

data_guide = pd.read_excel(DATA_GUIDE_PATH, sheet_name="2024_code_list")
collision_guide = data_guide[data_guide["table"] == "collision"].copy()
collision_guide.head()


,table,field name,code/format,label,note
0,collision,collision_index,NaN,NaN,unique value for each collision. The collision...
1,collision,collision_year,NaN,NaN,NaN
2,collision,collision_ref_no,NaN,NaN,In year id used by the police to reference a c...
3,collision,location_easting_osgr,NaN,NaN,Null if not known
4,collision,location_northing_osgr,NaN,NaN,Null if not known


In [4]:
df.head()


,collision_index,collision_year,collision_severity,number_of_vehicles,number_of_casualties,date,day_of_week,time,speed_limit
0,202517M102225,2025,3,1,1,15/02/2025,7,19:15,30
1,202417S111924,2024,2,1,1,22/10/2024,3,14:48,30
2,2025111687011,2025,3,2,1,19/12/2025,6,06:40,30
3,2025070326701,2025,1,1,1,22/04/2025,3,20:03,30
4,2025070353559,2025,3,2,1,28/04/2025,2,17:40,30


## Data Cleaning

In [5]:
def handle_missing_sentinels(df, sentinel=-1):
    """Convert STATS19's missing/unknown sentinel code to a proper NaN.

    STATS19 encodes "missing or unknown" on integer-coded fields as a literal
    ``-1`` value rather than a null. Left as-is, ``-1`` counts as "valid" data
    under pandas' native missing-value tooling — e.g. ``df.isna().sum()``
    would silently miss it, understating true missingness and risking biased
    downstream cleaning/EDA (a naive `.dropna()` would also just never fire).
    This function scans every integer-coded column for the sentinel and, where
    found, casts that column to a nullable integer dtype and replaces the
    sentinel with ``pd.NA`` so pandas' native null-handling works correctly.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with STATS19-coded integer columns.
    sentinel : int, default -1
        The STATS19 missing/unknown sentinel value.

    Returns
    -------
    pd.DataFrame
        Copy of ``df`` with the sentinel replaced by NaN on affected columns.
    """
    df = df.copy()
    int_cols = df.select_dtypes(include=["int8", "int16", "int32", "int64"]).columns
    for col in int_cols:
        if (df[col] == sentinel).any():
            df[col] = df[col].astype("Int64").replace(sentinel, pd.NA)
    return df


In [6]:
print('Missing values BEFORE sentinel handling (naive .isna(), will miss -1 sentinels):')
print(df.isna().sum())

df = handle_missing_sentinels(df)

print('\nMissing values AFTER sentinel handling (-1 now correctly counted as NaN):')
print(df.isna().sum())


Missing values BEFORE sentinel handling (naive .isna(), will miss -1 sentinels):
collision_index         0
collision_year          0
collision_severity      0
number_of_vehicles      0
number_of_casualties    0
date                    0
day_of_week             0
time                    0
speed_limit             0
dtype: int64

Missing values AFTER sentinel handling (-1 now correctly counted as NaN):
collision_index         0
collision_year          0
collision_severity      0
number_of_vehicles      0
number_of_casualties    0
date                    0
day_of_week             0
time                    0
speed_limit             3
dtype: int64
